# AMI Meeting Corpus — MCVAD IHM Değerlendirmesi
### Tez Hipotezi: Multi-Channel → Clustering'siz Diyarizasyon

**Temel varsayım:** `Headset-i = Konuşmacı-i` (fiziksel garanti)  
**Hedef:** Clustering olmadan Speaker Confusion ≈ 0% göstermek  
**Kalan problem:** Overlap tespiti (tezin asıl katkısı)

---
**Değişiklikler (v3):**
- `ihm_mode=True` → bleed suppression ve dominant-only KAPALI
- `overlap` etiketi Hungarian maliyet matrisinden çıkarıldı
- Overlap matrisi görsel olarak yazdırılıyor
- Ablation study hücresi eklendi

## Hücre 1 — Ortam Kurulumu

In [ ]:
import os, sys

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/meeting_analyzer'
    WORK_DIR     = '/content/ami'
    %pip install -q pyannote.audio==3.1.1 pyannote.metrics==3.2.1 pydub librosa soundfile scipy matplotlib
    !apt-get install -y -q ffmpeg
else:
    PROJECT_ROOT = os.getcwd()
    WORK_DIR     = os.path.join(PROJECT_ROOT, 'ami_data')

os.makedirs(WORK_DIR, exist_ok=True)
print(f'Ortam hazır.')
print(f'  PROJECT_ROOT : {PROJECT_ROOT}')
print(f'  WORK_DIR     : {WORK_DIR}')

## Hücre 2 — AMI IHM Ses Dosyalarını İndir

In [ ]:
import urllib.request

MEETINGS = ['EN2001a', 'EN2001b', 'EN2001e']
AMI_BASE = 'https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus'

for m in MEETINGS:
    ad = os.path.join(WORK_DIR, m, 'audio')
    os.makedirs(ad, exist_ok=True)
    for i in range(4):
        url = f'{AMI_BASE}/{m}/audio/{m}.Headset-{i}.wav'
        out = os.path.join(ad, f'{m}.Headset-{i}.wav')
        if not os.path.exists(out):
            try:
                urllib.request.urlretrieve(url, out)
                print(f'OK: {m} H{i}')
            except Exception as e:
                print(f'HATA: {m} H{i} → {e}')
        else:
            print(f'Mevcut: {m} H{i}')

## Hücre 3 — Referans RTTM İndir

In [ ]:
RTTM_FOLDERS = ['test', 'dev', 'train']
BASE_URL = 'https://raw.githubusercontent.com/BUTspeechFIT/AMI-diarization-setup/main/only_words/rttms'

for m in MEETINGS:
    rd = os.path.join(WORK_DIR, m, 'rttm')
    os.makedirs(rd, exist_ok=True)
    op = os.path.join(rd, f'{m}.rttm')
    if os.path.exists(op):
        print(f'Mevcut: {m}.rttm')
        continue
    for folder in RTTM_FOLDERS:
        try:
            urllib.request.urlretrieve(f'{BASE_URL}/{folder}/{m}.rttm', op)
            print(f'OK ({folder}): {m}.rttm')
            break
        except:
            continue

## Hücre 4 — MCVAD Yükle + IHM Config

> **`ihm_mode=True` nedir?**  
> Geleneksel modda MCVAD, kanallar arası enerji karşılaştırması yaparak  
> zayıf kanalı susturur. Bu IHM'de kanal kimliğini bozar (Confusion ↑).  
> `ihm_mode=True` ile kanallar tamamen bağımsız VAD uygular —  
> kanal kimliği hiç değişmez, sadece overlap tespiti yapılır.

In [ ]:
import importlib

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import module1_vad.config as config
importlib.reload(config)

# ── IHM Parametreleri ────────────────────────────────────────────────
# Her headset fiziksel olarak tek bir kişiye ait → kanallar arası
# enerji karşılaştırması YAPILMAZ.
config.ADAPTIVE_THRESHOLD_MULTIPLIER = 3.0
config.ADAPTIVE_WINDOW_SECONDS       = 30.0
config.MIN_SEGMENT_MS                = 300      # 500ms çok kaybettiriyordu
config.NOISE_FLOOR                   = 0.02
config.GLOBAL_SPEECH_FLOOR           = 0.03
config.IHM_OVERLAP_RATIO             = 0.30    # >=0.30*max → gerçek overlap

import module1_vad.energy_vad       as ev;  importlib.reload(ev)
import module1_vad.mcvad            as mc;  importlib.reload(mc)
import module1_vad.audio_standardizer as ast_mod; importlib.reload(ast_mod)
import module1_vad.rttm_writer      as rw;  importlib.reload(rw)

from module1_vad.mcvad             import MultiChannelVAD
from module1_vad.audio_standardizer import AudioStandardizer

print('Modüller yüklendi ✓')
print(f'  θ             = {config.ADAPTIVE_THRESHOLD_MULTIPLIER}')
print(f'  pencere       = {config.ADAPTIVE_WINDOW_SECONDS}s')
print(f'  min_segment   = {config.MIN_SEGMENT_MS}ms')
print(f'  overlap_ratio = {config.IHM_OVERLAP_RATIO}')
print(f'  IHM modu      = True  (bleed suppression + dominant-only KAPALI)')

## Hücre 5 — Yardımcı Fonksiyonlar

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment


def parse_ref_rttm(path):
    """Referans RTTM dosyasını okur, segment listesi döndürür."""
    segments = []
    with open(path) as f:
        for line in f:
            if not line.startswith('SPEAKER'):
                continue
            p = line.split()
            segments.append({
                'start': float(p[3]),
                'dur':   float(p[4]),
                'spk':   p[7]
            })
    return segments


def compute_temporal_overlap(pred_segments, ref_segments, pred_spk, ref_spk):
    """
    pred_spk etiketli tahmin ile ref_spk etiketli referans arasındaki
    zamansal örtüşmeyi (saniye) hesaplar.

    ÖNEMLİ: 'overlap' etiketli tahmin segmentleri KATILMAZ.
    Sadece type='single' olan segmentler kullanılır çünkü:
    - 'overlap' bir konuşmacı değil, durum etiketidir.
    - Overlap segmentleri için individual konuşmacılar speakers[] listesindedir.
    """
    pred_intervals = [
        (s['start'], s['start'] + s['duration'])
        for s in pred_segments
        if s.get('speaker') == pred_spk
        and s.get('type') != 'overlap'     # ← kritik filtre
    ]
    ref_intervals = [
        (s['start'], s['start'] + s['dur'])
        for s in ref_segments if s['spk'] == ref_spk
    ]
    overlap = 0.0
    for ps, pe in pred_intervals:
        for rs, re in ref_intervals:
            overlap += max(0.0, min(pe, re) - max(ps, rs))
    return overlap


def find_speaker_mapping(pred_segments, ref_rttm_path, verbose=True):
    """
    Hungarian Algoritması ile kanal → konuşmacı eşlemesini bulur.

    Düzeltmeler (v3):
    1. 'overlap' etiketi pred_speakers'dan çıkarılır — gerçek kanal değil.
    2. Single segmentler üzerinden overlap hesaplanır.
    3. Overlap matrisi yazdırılır (şeffaflık için).

    IHM prensibinde Headset-i = Konuşmacı-i (fiziksel garanti).
    ihm_mode=True ile kanal kimliği bozulmadığından Hungarian doğru çalışır.
    """
    ref_segs = parse_ref_rttm(ref_rttm_path)

    # 'overlap' etiketini filtrele — sadece gerçek kanal etiketleri
    pred_speakers = sorted(set(
        s['speaker'] for s in pred_segments
        if s['speaker'] != 'overlap'
    ))
    ref_speakers = sorted(set(s['spk'] for s in ref_segs))

    n_pred = len(pred_speakers)
    n_ref  = len(ref_speakers)
    n      = max(n_pred, n_ref)

    # Overlap matrisi (n_pred × n_ref)
    ov_mat = np.zeros((n_pred, n_ref))
    cost   = np.zeros((n, n))  # padded for Hungarian

    for i, ps in enumerate(pred_speakers):
        for j, rs in enumerate(ref_speakers):
            ov = compute_temporal_overlap(pred_segments, ref_segs, ps, rs)
            ov_mat[i, j] = ov
            cost[i, j]   = -ov  # minimize → maximize

    if verbose:
        # Overlap matrisini yazdır
        col_w = 9
        header = ' ' * 10 + ''.join(f'{s:>{col_w}}' for s in ref_speakers)
        print(f'  Overlap Matrisi (saniye):  [satır=tahmin, sütun=referans]')
        print(f'  {header}')
        for i, ps in enumerate(pred_speakers):
            row = ''.join(f'{ov_mat[i,j]:>{col_w}.1f}' for j in range(n_ref))
            print(f'  {ps:>8}: {row}')

    # Hungarian
    row_idx, col_idx = linear_sum_assignment(cost)

    mapping = {}
    if verbose:
        print(f'  Eşleme:')
    for r, c in zip(row_idx, col_idx):
        if r < n_pred and c < n_ref:
            ov  = ov_mat[r, c]
            flag = '✅' if ov > 60 else '⚠️'
            mapping[pred_speakers[r]] = ref_speakers[c]
            if verbose:
                print(f'    {flag} {pred_speakers[r]} → {ref_speakers[c]}  ({ov:.1f}s)')

    return mapping


def remap_and_write_rttm(segments, mapping, out_path, recording_id):
    """Tahmin segmentlerindeki konuşmacı etiketlerini yeniden adlandırıp RTTM yazar."""
    lines = []
    for seg in segments:
        dur = seg['duration']
        if dur <= 0:
            continue
        if seg['type'] == 'overlap':
            # Overlap: her konuşmacı için ayrı RTTM satırı
            for spk in seg.get('speakers', []):
                if spk == 'overlap':
                    continue
                mapped = mapping.get(spk, spk)
                lines.append(
                    f"SPEAKER {recording_id} 1 {seg['start']:.3f} {dur:.3f} "
                    f"<NA> <NA> {mapped} <NA> <NA>"
                )
        else:
            spk    = seg['speaker']
            mapped = mapping.get(spk, spk)
            lines.append(
                f"SPEAKER {recording_id} 1 {seg['start']:.3f} {dur:.3f} "
                f"<NA> <NA> {mapped} <NA> <NA>"
            )
    with open(out_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    return out_path


print('Yardımcı fonksiyonlar hazır ✓')

## Hücre 6 — IHM Pipeline: Per-Kanal VAD → Hungarian → RTTM

**Akış:**
```
Headset-i → bağımsız EnergyVAD → spk_i aktiflik vektörü
                                        ↓
                               Overlap tespiti (IHM)
                                        ↓
                        Hungarian (overlap filtreli) → eşleme
                                        ↓
                                  RTTM çıktısı
```

In [ ]:
import soundfile as sf
import time

all_segments = {}   # meeting → segment list (sonraki hücrelerde kullanılır)
all_mappings = {}   # meeting → mapping dict

for m in MEETINGS:
    print(f'\n{"="*62}')
    print(f'İŞLENİYOR: {m}')
    print(f'{"="*62}')

    audio_dir = os.path.join(WORK_DIR, m, 'audio')
    std = AudioStandardizer()

    # IHM modu:
    #   - bleed suppression KAPALI  (kanal kimliği korunur)
    #   - dominant-only    KAPALI  (kanal kimliği korunur)
    #   - overlap_ratio=0.30       (iki kanal >= 0.30*max → gerçek overlap)
    #   - use_spectral=False       (IHM headset için gereksiz)
    vad = MultiChannelVAD(
        ihm_mode=True,
        overlap_ratio=config.IHM_OVERLAP_RATIO,
        use_spectral=False,
    )

    # 1. Kanalları yükle
    channels = {}
    for i in range(4):
        raw  = os.path.join(audio_dir, f'{m}.Headset-{i}.wav')
        stdp = os.path.join(audio_dir, f'{m}.Headset-{i}.std.wav')
        if not os.path.exists(raw):
            print(f'  ⚠️  Headset-{i} bulunamadı, atlanıyor.')
            continue
        if not os.path.exists(stdp):
            std.standardize(raw, stdp)
        audio, sr = sf.read(stdp, dtype='float32')
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        channels[f'spk_{i}'] = audio.astype(np.float32)
        print(f'  🎤 Headset-{i}: {len(audio)/sr:.1f}s')

    if not channels:
        print('  ❌ Hiç kanal yüklenemedi, atlanıyor.')
        continue

    # 2. IHM-VAD çalıştır
    print(f'  ⏳ IHM-VAD çalışıyor...')
    t0   = time.time()
    segs = vad.process(channels)
    dt   = time.time() - t0

    n_single  = sum(1 for s in segs if s['type'] == 'single')
    n_overlap = sum(1 for s in segs if s['type'] == 'overlap')
    print(f'  ✅ {len(segs)} segment ({dt:.1f}s)')
    print(f'     single={n_single}  overlap={n_overlap}')

    # 3. Hungarian mapping ('overlap' filtreli)
    ref_rttm = os.path.join(WORK_DIR, m, 'rttm', f'{m}.rttm')
    print(f'\n  🔗 Speaker Mapping (overlap filtreli Hungarian):')
    mapping = find_speaker_mapping(segs, ref_rttm, verbose=True)

    # 4. RTTM yaz
    pred_rttm = os.path.join(WORK_DIR, m, 'rttm', f'{m}.pred.rttm')
    remap_and_write_rttm(segs, mapping, pred_rttm, recording_id=m)
    print(f'  💾 RTTM yazıldı → {pred_rttm}')

    all_segments[m] = segs
    all_mappings[m] = mapping

print(f'\n{"="*62}')
print('Pipeline tamamlandı.')

## Hücre 7 — DER Hesaplama (pyannote.metrics)

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate
from pyannote.core import Annotation, Segment as PySegment


def rttm_to_annotation(path):
    """RTTM → pyannote Annotation nesnesi."""
    ann = Annotation()
    with open(path) as f:
        for i, line in enumerate(f):
            if not line.startswith('SPEAKER'):
                continue
            p = line.split()
            start = float(p[3])
            dur   = float(p[4])
            spk   = p[7]
            ann[PySegment(start, start + dur), f't{i}'] = spk
    return ann


metric = DiarizationErrorRate(collar=0.25)

print(f'=================================================================')
print(f'{"Toplantı":<12} {"DER":>8} {"FA":>8} {"Miss":>8} {"Conf":>8}')
print(f'=================================================================')

der_results = {}
for m in MEETINGS:
    ref_path  = os.path.join(WORK_DIR, m, 'rttm', f'{m}.rttm')
    pred_path = os.path.join(WORK_DIR, m, 'rttm', f'{m}.pred.rttm')

    if not os.path.exists(pred_path):
        print(f'{m:<12} → pred.rttm bulunamadı, atlanıyor.')
        continue

    ref = rttm_to_annotation(ref_path)
    hyp = rttm_to_annotation(pred_path)
    d   = metric(ref, hyp, detailed=True)

    total    = d['total']
    fa_pct   = d['false alarm']      / total * 100 if total > 0 else 0
    miss_pct = d['missed detection'] / total * 100 if total > 0 else 0
    conf_pct = d['confusion']        / total * 100 if total > 0 else 0
    der_pct  = d['diarization error rate'] * 100

    der_results[m] = {'der': der_pct, 'fa': fa_pct, 'miss': miss_pct, 'conf': conf_pct}
    print(f'{m:<12} {der_pct:>7.1f}% {fa_pct:>7.1f}% {miss_pct:>7.1f}% {conf_pct:>7.1f}%')

print(f'=================================================================')
print(f'{"ORTALAMA DER":<12} {abs(metric)*100:>7.1f}%')
print(f'=================================================================')

## Hücre 8 — Sonuç Özeti + Tez Yorumu

In [ ]:
if der_results:
    avg_der  = np.mean([v['der']  for v in der_results.values()])
    avg_fa   = np.mean([v['fa']   for v in der_results.values()])
    avg_miss = np.mean([v['miss'] for v in der_results.values()])
    avg_conf = np.mean([v['conf'] for v in der_results.values()])

    print('╔══════════════════════════════════════════════════════════╗')
    print('║       MCVAD IHM v3 — AMI Sonuçları                     ║')
    print('╠══════════════════════════════════════════════════════════╣')
    print(f'║  Ortalama DER          : {avg_der:>6.1f}%                       ║')
    print(f'║  ├─ False Alarm (FA)   : {avg_fa:>6.1f}%                       ║')
    print(f'║  ├─ Missed Detection   : {avg_miss:>6.1f}%                       ║')
    print(f'║  └─ Speaker Confusion  : {avg_conf:>6.1f}%                       ║')
    print('║                                                          ║')
    print(f'║  Config: IHM modu, overlap_ratio={config.IHM_OVERLAP_RATIO:.2f}              ║')
    print(f'║  Mapping: Hungarian (overlap filtreli)                  ║')
    print('╠══════════════════════════════════════════════════════════╣')

    # Tez yorumu
    if avg_conf < 5:
        print('║  ✅ Confusion ≈ 0 → HİPOTEZ KANITLANDI:               ║')
        print('║     Multi-channel IHM, clustering olmadan konuşmacı    ║')
        print('║     kimliğini çözüyor (Headset-i = Konuşmacı-i)        ║')
        print('║  ➡️  Kalan DER = FA + Miss → Overlap tespiti problemi. ║')
    elif avg_conf < 15:
        print('║  🟡 Confusion düşük ama sıfır değil.                   ║')
        print('║     overlap_ratio parametresini dene: 0.20 – 0.40      ║')
    else:
        print('║  🔴 Confusion hâlâ yüksek — kök neden analizi gerekli. ║')

    print('╚══════════════════════════════════════════════════════════╝')

## Hücre 9 — Ablation Study: overlap_ratio Parametresi

Tezin katkısı: `overlap_ratio` değeri overlap tespitini nasıl etkiliyor?
- **0.20**: Daha fazla overlap kabul → FA↑, Conf↓  
- **0.50**: Daha az overlap kabul → FA↓, Miss↑

In [ ]:
ablation_ratios = [0.10, 0.20, 0.30, 0.40, 0.50]

ablation_results = []

for ratio in ablation_ratios:
    partial_ders = []
    for m in MEETINGS:
        audio_dir = os.path.join(WORK_DIR, m, 'audio')
        std = AudioStandardizer()

        vad = MultiChannelVAD(
            ihm_mode=True,
            overlap_ratio=ratio,
            use_spectral=False,
        )

        channels = {}
        for i in range(4):
            stdp = os.path.join(audio_dir, f'{m}.Headset-{i}.std.wav')
            if not os.path.exists(stdp):
                continue
            audio, sr = sf.read(stdp, dtype='float32')
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            channels[f'spk_{i}'] = audio.astype(np.float32)

        if not channels:
            continue

        segs = vad.process(channels)
        ref_rttm  = os.path.join(WORK_DIR, m, 'rttm', f'{m}.rttm')
        mapping   = find_speaker_mapping(segs, ref_rttm, verbose=False)
        abl_rttm  = os.path.join(WORK_DIR, m, 'rttm', f'{m}.abl_{ratio:.2f}.rttm')
        remap_and_write_rttm(segs, mapping, abl_rttm, recording_id=m)

        abl_metric = DiarizationErrorRate(collar=0.25)
        ref = rttm_to_annotation(ref_rttm)
        hyp = rttm_to_annotation(abl_rttm)
        d   = abl_metric(ref, hyp, detailed=True)
        partial_ders.append(d)

    if partial_ders:
        avg_d = np.mean([x['diarization error rate'] * 100 for x in partial_ders])
        avg_f = np.mean([x['false alarm'] / x['total'] * 100 for x in partial_ders])
        avg_m = np.mean([x['missed detection'] / x['total'] * 100 for x in partial_ders])
        avg_c = np.mean([x['confusion'] / x['total'] * 100 for x in partial_ders])
        ablation_results.append((ratio, avg_d, avg_f, avg_m, avg_c))

print(f'=================================================================')
print(f'ABLATION STUDY: overlap_ratio parametresi')
print(f'=================================================================')
print(f'{"ratio":>8} {"DER":>8} {"FA":>8} {"Miss":>8} {"Conf":>8}')
print(f'-----------------------------------------------------------------')
for ratio, der, fa, miss, conf in ablation_results:
    marker = ' ←' if ratio == config.IHM_OVERLAP_RATIO else ''
    print(f'{ratio:>8.2f} {der:>7.1f}% {fa:>7.1f}% {miss:>7.1f}% {conf:>7.1f}%{marker}')
print(f'=================================================================')

## Hücre 10 — v2 (Eski) vs v3 (IHM) Karşılaştırması

Tezdeki Tablo 1: iki sistemin DER bileşenlerini karşılaştır.

In [ ]:
# Eski v2 sonuçları (manually kaydedildi — hücre başında notebook çalıştırılmadan önce)
v2_results = {
    'EN2001a': {'der': 77.9, 'fa': 11.9, 'miss': 7.3, 'conf': 58.6},
    'EN2001b': {'der': 73.0, 'fa': 14.2, 'miss': 7.7, 'conf': 51.1},
    'EN2001e': {'der': 57.1, 'fa': 12.0, 'miss': 10.3, 'conf': 34.8},
}
v2_avg = {k: np.mean([v2_results[m][k] for m in MEETINGS]) for k in ['der','fa','miss','conf']}

if der_results:
    v3_avg = {k: np.mean([der_results[m][k] for m in der_results]) for k in ['der','fa','miss','conf']}

    print(f'=================================================================')
    print(f'KARŞILAŞTIRMA: MCVAD v2 (eski) vs MCVAD IHM v3 (yeni)')
    print(f'=================================================================')
    print(f'{"Sistem":<20} {"DER":>8} {"FA":>8} {"Miss":>8} {"Conf":>8}')
    print(f'-----------------------------------------------------------------')
    print(f'{"MCVAD v2 (bleed+dom)":<20} {v2_avg["der"]:>7.1f}% {v2_avg["fa"]:>7.1f}% {v2_avg["miss"]:>7.1f}% {v2_avg["conf"]:>7.1f}%')
    print(f'{"MCVAD IHM v3":<20} {v3_avg["der"]:>7.1f}% {v3_avg["fa"]:>7.1f}% {v3_avg["miss"]:>7.1f}% {v3_avg["conf"]:>7.1f}%')
    print(f'=================================================================')

    conf_drop = v2_avg['conf'] - v3_avg['conf']
    der_drop  = v2_avg['der']  - v3_avg['der']
    print(f'\n  Confusion düşüşü : {conf_drop:+.1f} puan')
    print(f'  DER düşüşü       : {der_drop:+.1f} puan')
    print()
    if v3_avg['conf'] < 5:
        print('  ✅ Tez hipotezi kanıtlandı:')
        print('     Kanal=Konuşmacı garantisiyle Confusion ≈ 0%')
        print('     Kalan DER = FA + Miss → Overlap tespiti problemi.')
    elif conf_drop > 20:
        print('  🟡 Büyük iyileşme var ama Confusion hâlâ tam sıfır değil.')
        print('     overlap_ratio veya VAD eşiğini ayarla.')